# Medallion reporting pipeline (local Spark → Fabric OneLake)

**How to run:** set `FABRIC_TENANT_ID`, `FABRIC_CLIENT_ID`, `FABRIC_CLIENT_SECRET`
(PowerShell: `$env:FABRIC_TENANT_ID = "..."`) or edit **local_spark_session**, then
**Run All** in Jupyter or VS Code. Do not use `build_local_pipeline_notebook.py` to execute
the pipeline—that script only rebuilds this file from the split notebooks.

**Pipeline order:** HC → Okta → Cursor usage (Silver) → Fact Cursor Active (Gold)

Each code cell starts with `# Cell: <name>` and has matching markdown headers.


## medallion_config


In [ ]:
# Cell: medallion_config
"""Catalog, table, and business constants for Bronze / Silver / Gold notebooks."""

# --- Lakehouse / warehouse catalogs ---

WORKSPACEID  = "4cecee03-0629-4081-8ae8-b749ee04d43f"
LAKEHOUSE_WS = "4cecee03-0629-4081-8ae8-b749ee04d43f"

BRONZE_DATABASE = "AI_Medallion_Bronze"
SILVER_DATABASE = "AI_Medallion"
GOLD_DATABASE = "AI_Medallion"

SILVER_SCHEMA = "Silver"
GOLD_SCHEMA = "Gold"

BRONZE_MEDALLION_DATABASE = "AI_Medallion"
BRONZE_CURSOR_SCHEMA = "Bronze"

# Reference / lookup
COUNTRY_DATABASE = "AI_Medallion_Silver"
COUNTRY_TABLE = "CoreCountry_Excel_Hours_Historic"

# --- Bronze sources ---
TABLE_HC_BRONZE = f"{BRONZE_DATABASE}.dbo.HC_Bronze_Historical"
TABLE_OKTA_BRONZE = f"{BRONZE_DATABASE}.dbo.CoreOktaUser"
TABLE_CURSOR_BRONZE = f"{BRONZE_MEDALLION_DATABASE}.{BRONZE_CURSOR_SCHEMA}.cursor_usage"
TABLE_COUNTRY = f"{COUNTRY_DATABASE}.dbo.{COUNTRY_TABLE}"

# --- Silver targets / sources ---
TABLE_HC_SILVER = f"{SILVER_DATABASE}.{SILVER_SCHEMA}.HC_Silver_Historical"
TABLE_OKTA_SILVER = f"{SILVER_DATABASE}.{SILVER_SCHEMA}.Okta_User_for_AI"
TABLE_CURSOR_SILVER = f"{SILVER_DATABASE}.{SILVER_SCHEMA}.cursor_usage"

# --- Gold targets ---
TABLE_FACT_CURSOR_ACTIVE = f"{GOLD_DATABASE}.{GOLD_SCHEMA}.Fact_Cursor_Active"

# --- Shared business rules ---
NO_COUNTRY = "No Country"
HC_DIVISION_INC_DEFAULT = "Others"

# --- FactCursorActive (Gold) constants ---
CURSOR_ASSET_ID = "PENDING"
CURSOR_ASSET_NAME = "Cursor"
CURSOR_USE_CASE_CODE = "UC-9006-CURS-001"
CURSOR_USE_CASE_DISPLAY_NAME = "Active"
CURSOR_USE_CASE_MULTIPLIER = 1
CURSOR_AUTOMATION_POC = 1
CURSOR_TOTAL_HOURS_SAVED_POC = 0

# Incremental merge: one fact row per silver usage event (stable hash of silver grain)
FACT_CURSOR_ACTIVE_MERGE_KEY = "RecordSurrogateKey"



## local_spark_session


In [ ]:
# Cell: local_spark_session
import os
from typing import Optional

FABRIC_TENANT_ID = os.environ.get("FABRIC_TENANT_ID", "<TENANT_ID>")
FABRIC_CLIENT_ID = os.environ.get("FABRIC_CLIENT_ID", "<APP_CLIENT_ID>")
FABRIC_CLIENT_SECRET = os.environ.get("FABRIC_CLIENT_SECRET", "<APP_CLIENT_SECRET>")

LAKEHOUSE_AI_MEDALLION_ID = os.environ.get(
    "LAKEHOUSE_AI_MEDALLION_ID", "1ee46463-2495-44f5-9dc9-d635d0067483"
)
LAKEHOUSE_AI_MEDALLION_BRONZE_ID = os.environ.get(
    "LAKEHOUSE_AI_MEDALLION_BRONZE_ID", "36e5350a-61f6-4120-8c32-a38f55722907"
)
LAKEHOUSE_AI_MEDALLION_SILVER_ID = os.environ.get(
    "LAKEHOUSE_AI_MEDALLION_SILVER_ID", LAKEHOUSE_AI_MEDALLION_ID
)

ONELAKE_HOST = "onelake.dfs.fabric.microsoft.com"
DELTA_PACKAGES = "io.delta:delta-spark_2.12:3.2.0"
AZURE_PACKAGES = "org.apache.hadoop:hadoop-azure:3.3.6,com.azure:azure-storage-blob:12.25.1"

CURSOR_BRONZE_FILENAME: Optional[str] = os.environ.get(
    "CURSOR_BRONZE_FILENAME",
    "team-usage-events-24970505-2026-09-25.csv",
)

def _configure_onelake_oauth(builder):
    prefix = "fs.azure.account"
    host = ONELAKE_HOST
    return (
        builder.config(f"{prefix}.auth.type.{host}", "OAuth")
        .config(
            f"{prefix}.oauth.provider.type.{host}",
            "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
        )
        .config(f"{prefix}.oauth2.client.id.{host}", FABRIC_CLIENT_ID)
        .config(f"{prefix}.oauth2.client.secret.{host}", FABRIC_CLIENT_SECRET)
        .config(
            f"{prefix}.oauth2.client.endpoint.{host}",
            f"https://login.microsoftonline.com/{FABRIC_TENANT_ID}/oauth2/token",
        )
    )

def build_local_spark(app_name: str = "MedallionLocal"):
    from pyspark.sql import SparkSession

    builder = (
        SparkSession.builder.appName(app_name)
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config(
            "spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog",
        )
        .config("spark.jars.packages", f"{DELTA_PACKAGES},{AZURE_PACKAGES}")
    )
    builder = _configure_onelake_oauth(builder)
    return builder.getOrCreate()

def onelake_tables_root(lakehouse_id: str) -> str:
    return f"abfss://{WORKSPACEID}@{ONELAKE_HOST}/{lakehouse_id}.Lakehouse/Tables"

def register_delta_table_if_missing(spark, table_fqn: str, abfss_path: str) -> None:
    if spark.catalog.tableExists(table_fqn):
        return
    spark.sql(
        f"CREATE TABLE IF NOT EXISTS {table_fqn} USING DELTA LOCATION '{abfss_path}'"
    )

def register_pipeline_tables(spark) -> None:
    mappings = {
        TABLE_CURSOR_BRONZE: f"{onelake_tables_root(LAKEHOUSE_AI_MEDALLION_ID)}/Bronze/cursor_usage",
        TABLE_HC_BRONZE: f"{onelake_tables_root(LAKEHOUSE_AI_MEDALLION_BRONZE_ID)}/dbo/HC_Bronze_Historical",
        TABLE_OKTA_BRONZE: f"{onelake_tables_root(LAKEHOUSE_AI_MEDALLION_BRONZE_ID)}/dbo/CoreOktaUser",
        TABLE_COUNTRY: f"{onelake_tables_root(LAKEHOUSE_AI_MEDALLION_SILVER_ID)}/dbo/CoreCountry_Excel_Hours_Historic",
        TABLE_HC_SILVER: f"{onelake_tables_root(LAKEHOUSE_AI_MEDALLION_ID)}/Silver/HC_Silver_Historical",
        TABLE_OKTA_SILVER: f"{onelake_tables_root(LAKEHOUSE_AI_MEDALLION_ID)}/Silver/Okta_User_for_AI",
        TABLE_CURSOR_SILVER: f"{onelake_tables_root(LAKEHOUSE_AI_MEDALLION_ID)}/Silver/cursor_usage",
        TABLE_FACT_CURSOR_ACTIVE: f"{onelake_tables_root(LAKEHOUSE_AI_MEDALLION_ID)}/Gold/Fact_Cursor_Active",
    }
    for fqn, path in mappings.items():
        register_delta_table_if_missing(spark, fqn, path)

try:
    spark  # type: ignore[name-defined]
    _USING_ATTACHED_FABRIC_SESSION = True
except NameError:
    spark = build_local_spark()
    _USING_ATTACHED_FABRIC_SESSION = False

if not _USING_ATTACHED_FABRIC_SESSION:
    register_pipeline_tables(spark)

print(
    f"Spark {spark.version}; Fabric-attached session={_USING_ATTACHED_FABRIC_SESSION}"
)



## medallion_io


In [ ]:
# Cell: medallion_io
from pyspark.sql import DataFrame

def write_full_table(df: DataFrame, table_name: str) -> None:
    """Full load (overwrite) for Silver / dimension-style Gold tables."""
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )


def merge_incremental(
    spark,
    df: DataFrame,
    table_name: str,
    merge_key: str,
) -> None:
    """Incremental upsert on a single merge key column (Delta Lake)."""
    if not spark.catalog.tableExists(table_name):
        write_full_table(df, table_name)
        return

    from delta.tables import DeltaTable

    target = DeltaTable.forName(spark, table_name)
    (
        target.alias("target")
        .merge(
            df.alias("source"),
            f"target.{merge_key} = source.{merge_key}",
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )



## medallion_transforms


In [ ]:
# Cell: medallion_transforms
from pyspark.sql import Column, DataFrame
from pyspark.sql import functions as F

# from medallion_config import NO_COUNTRY


def trim_lower(col_name: str) -> Column:
    return F.lower(F.trim(F.col(col_name)))


def trim_col(col_name: str) -> Column:
    return F.trim(F.col(col_name))


def is_valid_email(col: Column) -> Column:
    """
    Validates an email using:
    - not null
    - not blank
    - exactly one @
    - no spaces
    - local part and domain are non-empty
    - domain contains a dot
    """
    normalized = F.lower(F.trim(col))

    return (
        col.isNotNull()
        & (F.trim(col) != "")
        & normalized.rlike(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")
    )

def date_from_yyyymm(month_no_col: Column) -> Column:
    """First day of month from YYYYMM integer column."""
    year = (month_no_col / F.lit(100)).cast("int")
    month = (month_no_col % F.lit(100)).cast("int")
    return F.make_date(year, month, F.lit(1))


def load_country_lookup(spark, country_table: str) -> DataFrame:
    return spark.table(country_table).select(
        F.col("Country").alias("_country_key"),
        F.col("DisplayName").alias("_country_display"),
    )


def with_normalized_country(
    df: DataFrame,
    country_source_col: str,
    country_lookup: DataFrame,
    output_col: str = "OktaUserCountry",
) -> DataFrame:
    return (
        df.join(
            country_lookup,
            F.col(country_source_col) == F.col("_country_key"),
            "left",
        )
        .withColumn(
            output_col,
            F.coalesce(F.col("_country_display"), F.lit(NO_COUNTRY)),
        )
        .drop("_country_key", "_country_display")
    )


def cursor_usage_record_key() -> Column:
    """Stable key for incremental gold loads (one key per silver CursorUsage row)."""
    parts = [
        F.coalesce(F.col("CursorUsageUser"), F.lit("")),
        F.coalesce(F.col("CursorUsageDate").cast("string"), F.lit("")),
        F.coalesce(F.col("CursorUsageKind"), F.lit("")),
        F.coalesce(F.col("CursorUsageModel"), F.lit("")),
        F.coalesce(F.col("CursorUsageTotalTokens").cast("string"), F.lit("")),
        F.coalesce(F.col("CursorUsageCost").cast("string"), F.lit("")),
    ]
    return F.sha2(F.concat_ws("|", *parts), 256)



## helpers


In [ ]:
# Cell: helpers
def show_sample(df, n: int = 10) -> None:
    try:
        display(df.limit(n))  # Fabric / IPython
    except NameError:
        df.limit(n).show(truncate=False)



## hc_historical_bronze_to_silver


In [ ]:
# Cell: hc_historical_bronze_to_silver
bronze = spark.table(TABLE_HC_BRONZE)
country_lookup = load_country_lookup(spark, TABLE_COUNTRY)

hc = bronze.select(
    F.col("`Month`").cast("int").alias("Month_No"),
    F.col("`Person ID`").alias("OktaUserEmployeeNumber"),
    F.col("`Employee Code`").alias("HCEmployeeNumber"),
    F.col("`Fullname [Employee]`").alias("OktaUserFullName"),
    F.col("`Region [Employee]`").alias("HCRegion"),
    F.col("`Country [Employee]`").alias("_raw_country"),
    F.col("`Office [Employee]`").alias("HCOffice"),
    F.col("`Business Unit [Employee]`").alias("HCBusinessUnit"),
    F.col("`Division [Employee]`").alias("HCDivision"),
    F.col("`Department [Employee]`").alias("HCDepartment"),
    F.col("`Legal Entity [Employee]`").alias("HCLegalEntity"),
    F.col("`Grid Subgroup [Employee]`").alias("HCGridSubgroup"),
    F.col("`Talent Segment [Employee]`").alias("HCTalentSegment"),
    F.col("`Job Title [Employee]`").alias("HCJobTitle"),
    F.lower(F.trim(F.col("`Email [Employee]`"))).alias("OktaUserEmail"),
    F.col("`User Network [Employee]`").alias("HCUserNetwork"),
    F.col("`Inc. Roles`").alias("HCIncRoles"),
    F.col("`Okta Region`").alias("OktaUserRegion"),
    F.col("`Division INC`").alias("HCDivisionINC"),
    F.col("`Total Employees`").cast("int").alias("HCTotalEmployees"),
    F.col("`Area Emeal`").alias("HCAreaEmeal"),
    F.col("KeyIdEmployeeCountry").alias("HCKeyIdEmployeeCountry"),
    F.col("`Business Unit Code [Employee]`").alias("HCBusinessUnitCode"),
    F.col("`Replace Mapping`").alias("HCReplaceMapping"),
)

hc = with_normalized_country(hc, "_raw_country", country_lookup).drop("_raw_country")

hc = hc.withColumn("HCDate", date_from_yyyymm(F.col("Month_No")))
hc = hc.withColumn(
    "HCKeyDivision",
    F.concat(F.col("HCDivisionINC"), F.lit("/"), F.col("OktaUserCountry")),
)
hc = hc.withColumn(
    "HCKeyEmail",
    F.concat(F.col("Month_No").cast("string"), F.lit("/"), F.col("OktaUserEmail")),
)

# Business keys: Month_No + OktaUserEmail should be unique per spec; enforce not null on required fields
hc = hc.filter(
    F.col("Month_No").isNotNull()
    & F.col("OktaUserEmail").isNotNull()
    & F.col("HCDate").isNotNull()
    & F.col("HCKeyDivision").isNotNull()
    & F.col("HCKeyEmail").isNotNull()
)

write_full_table(hc, TABLE_HC_SILVER)

show_sample(hc)



## okta_user_bronze_to_silver


In [ ]:
# Cell: okta_user_bronze_to_silver
bronze = spark.table(TABLE_OKTA_BRONZE)
country_lookup = load_country_lookup(spark, TABLE_COUNTRY)

login_norm = F.lower(F.trim(F.col("login")))
email_norm = F.lower(F.trim(F.col("email")))

okta = bronze.select(
    F.col("id").alias("OktaUserId"),
    F.trim(F.col("display_name")).alias("OktaUserFullName"),
    F.trim(F.col("first_name")).alias("OktaUserFirstName"),
    F.trim(F.col("last_name")).alias("OktaUserLastName"),
    login_norm.alias("OktaUserEmail"),
    email_norm.alias("OktaUserFullEmail"),
    F.trim(F.col("employee_number")).alias("OktaUserEmployeeNumber"),
    F.trim(F.col("region")).alias("OktaUserRegion"),
    F.col("okta_id").alias("OktaUserOktaId"),
    F.col("updated_at").cast("timestamp").alias("OktaUserUpdatedAt"),
    F.col("deleted_at").cast("timestamp").alias("OktaUserDeletedAt"),
    F.trim(F.col("country")).alias("_raw_country"),
    F.trim(F.col("okta_tenant")).alias("OktaUserTenant"),
)

okta = okta.filter(
    F.col("OktaUserId").isNotNull()
    & is_valid_email(F.col("OktaUserEmail"))
    & is_valid_email(F.col("OktaUserFullEmail"))
    & (F.lower(F.col("OktaUserRegion")) != F.lit("japan-onentt"))
    & (F.lower(F.col("OktaUserTenant")) != F.lit("connect-onentt"))
)

okta = with_normalized_country(okta, "_raw_country", country_lookup).drop("_raw_country")

write_full_table(okta, TABLE_OKTA_SILVER)

show_sample(okta)



## cursor_usage_bronze_to_silver


In [ ]:
# Cell: cursor_usage_bronze_to_silver
bronze_df = spark.table(TABLE_CURSOR_BRONZE)
if CURSOR_BRONZE_FILENAME:
    bronze_df = bronze_df.filter(F.col("filename") == CURSOR_BRONZE_FILENAME)
bronze = bronze_df

okta = spark.table(TABLE_OKTA_SILVER)
hc = spark.table(TABLE_HC_SILVER)

user_norm = F.lower(F.trim(F.col("User")))

base = (
    bronze.filter(is_valid_email(F.col("User")))
    .select(
        F.col("Date").cast("date").alias("CursorUsageDate"),
        user_norm.alias("CursorUsageUser"),
        F.col("`Service Account Name`").alias("CursorUsageServiceAccountName"),
        F.col("`Service Account ID`").alias("CursorUsageServiceAccountID"),
        F.col("`Cloud Agent ID`").alias("CursorUsageCloudAgentID"),
        F.col("`Automation ID`").alias("CursorUsageAutomationID"),
        F.col("Kind").alias("CursorUsageKind"),
        F.col("Model").alias("CursorUsageModel"),
        F.col("`Max Mode`").alias("CursorUsageMaxMode"),
        F.col("`Input (w/ Cache Write)`").cast("long").alias("CursorUsageInputWithCacheWrite"),
        F.col("`Input (w/o Cache Write)`").cast("long").alias("CursorUsageInputWithoutCacheWrite"),
        F.col("`Cache Read`").cast("long").alias("CursorUsageCacheRead"),
        F.col("`Output Tokens`").cast("long").alias("CursorUsageOutputTokens"),
        F.col("`Total Tokens`").cast("long").alias("CursorUsageTotalTokens"),
        F.col("Cost").cast("decimal(18,8)").alias("CursorUsageCost"),
        F.col("FileName").alias("CursorUsageFileName"),
        F.col("ExecutionDate").cast("timestamp").alias("CursorUsageIngestionDate"),
    )
    .withColumn(
        "CursorUsageMonthNo",
        (F.year("CursorUsageDate") * F.lit(100) + F.month("CursorUsageDate")).cast("int"),
    )
)

okta_slim = okta.select(
    "OktaUserId",
    "OktaUserEmail",
    "OktaUserFullEmail",
    "OktaUserEmployeeNumber",
    "OktaUserFullName",
    "OktaUserRegion",
    "OktaUserCountry",
)

joined = base.join(
    okta_slim,
    (F.col("CursorUsageUser") == F.col("OktaUserEmail"))
    | (F.col("CursorUsageUser") == F.col("OktaUserFullEmail")),
    "inner",
)

joined = joined.withColumn(
    "CursorUsageKeyEmail",
    F.when(
        F.col("OktaUserEmail").isNotNull(),
        F.concat(
            F.col("CursorUsageMonthNo").cast("string"),
            F.lit("/"),
            F.col("OktaUserEmail"),
        ),
    ),
).withColumn(
    "CursorUsageKeyFullEmail",
    F.when(
        F.col("OktaUserFullEmail").isNotNull(),
        F.concat(
            F.col("CursorUsageMonthNo").cast("string"),
            F.lit("/"),
            F.col("OktaUserFullEmail"),
        ),
    ),
)

hc_slim = hc.select(
    F.col("HCKeyEmail"),
    F.col("HCBusinessUnit"),
    F.col("HCBusinessUnitCode"),
    F.col("HCDivision"),
    F.col("HCJobTitle"),
    F.col("HCOffice"),
    F.col("HCTalentSegment"),
    F.col("Month_No").alias("HCMonthNo"),
    F.col("OktaUserCountry").alias("HCOktaUserCountry"),
    F.col("HCDivisionINC"),
)

joined = joined.join(
    hc_slim,
    (F.col("HCKeyEmail") == F.col("CursorUsageKeyEmail"))
    | (F.col("HCKeyEmail") == F.col("CursorUsageKeyFullEmail")),
    "left",
).withColumn(
    "HCDivisionINC",
    F.coalesce(F.col("HCDivisionINC"), F.lit(HC_DIVISION_INC_DEFAULT)),
)

write_df = joined.distinct()
write_df = write_df.withColumn(
    FACT_CURSOR_ACTIVE_MERGE_KEY,
    cursor_usage_record_key(),
)

merge_incremental(
    spark,
    write_df,
    TABLE_CURSOR_SILVER,
    FACT_CURSOR_ACTIVE_MERGE_KEY,
)

show_sample(write_df)



## cursor_usage_gold


In [ ]:
# Cell: cursor_usage_gold
silver = spark.table(TABLE_CURSOR_SILVER)

request = F.lit(1).cast("int")
use_case_multiplier = F.lit(CURSOR_USE_CASE_MULTIPLIER).cast("decimal(18,8)")
automation_poc = F.lit(CURSOR_AUTOMATION_POC).cast("decimal(18,8)")
total_hours_saved_poc = F.lit(CURSOR_TOTAL_HOURS_SAVED_POC).cast("decimal(18,8)")

fact = silver.select(
    F.col("CursorUsageMonthNo").alias("Month_No"),
    request.alias("Request"),
    F.lit(CURSOR_ASSET_ID).alias("assetId"),
    F.lit(CURSOR_ASSET_NAME).alias("assetName"),
    F.lit(None).cast("string").alias("eventType"),
    F.lit(None).cast("string").alias("id_project"),
    F.lit(None).cast("string").alias("project"),
    F.lit(None).cast("string").alias("account"),
    F.col("OktaUserId").alias("userId"),
    F.col("OktaUserFullName").alias("username"),
    F.col("OktaUserEmployeeNumber").alias("employeeid"),
    F.col("OktaUserEmail").alias("userEmail"),
    F.col("OktaUserFullEmail").alias("userFullEmail"),
    F.col("OktaUserRegion").alias("Division"),
    F.col("OktaUserCountry").alias("userCountry"),
    F.lit(CURSOR_USE_CASE_CODE).alias("useCaseCode"),
    F.lit(CURSOR_USE_CASE_DISPLAY_NAME).alias("useCaseDisplayName"),
    (request.cast("decimal(18,8)") * total_hours_saved_poc * use_case_multiplier).alias(
        "UseCaseHoursSaved"
    ),
    (
        request.cast("decimal(18,8)")
        * (total_hours_saved_poc / automation_poc)
    ).alias("UseCaseTotalHours"),
    F.lit(1).cast("int").alias("UserWithActivity"),
    F.col("CursorUsageDate").alias("LastModification"),
    F.col("HCBusinessUnit").alias("User_UN"),
    F.col("HCJobTitle").alias("JobRoles"),
    F.col("HCOktaUserCountry").alias("Employee_Country"),
    F.col("HCOffice").alias("Employee_Office"),
    F.col("HCTalentSegment").alias("Segment"),
    F.col("HCDivision").alias("Foundation"),
    F.col("HCBusinessUnitCode").alias("Employee_Business_Unit_Code"),
    F.lit(None).cast("string").alias("asset_model"),
    F.lit(None).cast("string").alias("asset_useCase"),
    F.lit(None).cast("string").alias("asset_mySourceType"),
    F.lit(None).cast("string").alias("asset_sourceType"),
    F.lit(None).cast("string").alias("billingClientName"),
    F.lit(None).cast("string").alias("billingCountry"),
    F.lit(None).cast("string").alias("billingRegion"),
    F.lit(None).cast("string").alias("flowid"),
    use_case_multiplier.alias("useCaseMultiplier"),
    automation_poc.alias("automation_poc"),
    total_hours_saved_poc.alias("totalHoursSaved_poc"),
    F.col(FACT_CURSOR_ACTIVE_MERGE_KEY),
)

# Required not-null columns per spec
fact = fact.filter(
    F.col("Month_No").isNotNull()
    & F.col("Request").isNotNull()
    & F.col("assetId").isNotNull()
    & F.col("assetName").isNotNull()
    & F.col("useCaseCode").isNotNull()
    & F.col("useCaseDisplayName").isNotNull()
    & F.col("UseCaseHoursSaved").isNotNull()
    & F.col("UseCaseTotalHours").isNotNull()
    & F.col("UserWithActivity").isNotNull()
    & F.col("LastModification").isNotNull()
    & F.col("useCaseMultiplier").isNotNull()
    & F.col("automation_poc").isNotNull()
    & F.col("totalHoursSaved_poc").isNotNull()
    & F.col(FACT_CURSOR_ACTIVE_MERGE_KEY).isNotNull()
)

merge_incremental(
    spark,
    fact,
    TABLE_FACT_CURSOR_ACTIVE,
    FACT_CURSOR_ACTIVE_MERGE_KEY,
)

show_sample(fact)

